In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
import torchvision.datasets as datasets
import torchvision.transforms as transforms
print(torch.__version__)

2.12.0+cpu


In [5]:
import pip

def install(package):
    if hasattr(pip, 'main'):
        pip.main(['install', package])
    else:
        pip._internal.main(['install', package])

install('torch-directml')

Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.


ERROR: Could not find a version that satisfies the requirement torch-directml (from versions: none)


ERROR: No matching distribution found for torch-directml


In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # Среднее и стандартное отклонение MNIST
])

# 2. Скачивание и создание объектов датасета
train_dataset = datasets.MNIST(
    root='./data', 
    train=True, 
    transform=transform, 
    download=True
)

test_dataset = datasets.MNIST(
    root='./data', 
    train=False, 
    transform=transform, 
    download=True
)

# 3. Создание загрузчиков данных (DataLoader) для итерации по батчам
train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

# Проверка размерности первого батча
images, labels = next(iter(train_loader))
print(f"Размерность тензора изображений: {images.shape}")  # [64, 1, 28, 28] -> [батч, каналы, высота, ширина]
print(f"Размерность тензора меток: {labels.shape}")   

Размерность тензора изображений: torch.Size([64, 1, 28, 28])
Размерность тензора меток: torch.Size([64])


In [3]:
#print(images[0,0,:,:])

In [4]:
# ==========================================
# 2. АРХИТЕКТУРА МОДЕЛИ (MLP)
# ==========================================
class MultilayerPerceptron(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(MultilayerPerceptron, self).__init__()
        # Входной слой -> Скрытый слой
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.gelu = nn.GELU()
        # Скрытый слой -> Выходной слой
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = x.view(x.size(0), -1) 
        out = self.fc1(x)
        out = self.gelu(out)
        out = self.fc2(out)  #Логиты на выходе (без Softmax, так как он внутри CrossEntropyLoss)
        return out




In [5]:
# Инициализируем модель (4 входа, 16 нейронов в скрытом слое, 3 выхода)
model = MultilayerPerceptron(input_dim=784, hidden_dim=256, output_dim=10)


# ==========================================
# 3. ФУНКЦИЯ ПОТЕРЬ И ОПТИМИЗАТОР
# ==========================================
criterion = nn.CrossEntropyLoss()  # Подходит для многоклассовой классификации
optimizer = optim.Adam(model.parameters(), lr=0.01)

# ==========================================
# 4. ЦИКЛ ОБУЧЕНИЯ (TRAINING LOOP)
# ==========================================
epochs = 40

for epoch in range(epochs):
    model.train()  # Перевод модели в режим обучения
    epoch_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()  # Сброс градиентов

        outputs = model(images)  # Прямой проход
        loss = criterion(outputs, labels)  # Расчет ошибки

        loss.backward()  # Обратный проход (вычисление градиентов)
        optimizer.step()  # Обновление весов

        epoch_loss += loss.item()

    # Выводим метрики каждые 10 эпох
    if (epoch + 1) % 10 == 0:
        print(
            f"Эпоха {epoch+1:02d}/{epochs} | Средняя ошибка батча: {epoch_loss/len(train_loader):.4f}"
        )


Эпоха 10/40 | Средняя ошибка батча: 0.1521
Эпоха 20/40 | Средняя ошибка батча: 0.1382
Эпоха 30/40 | Средняя ошибка батча: 0.1232
Эпоха 40/40 | Средняя ошибка батча: 0.1115


In [14]:
# ==========================================
# 5. ТЕСТИРОВАНИЕ И ОЦЕНКА ТОЧНОСТИ
# ==========================================
model.eval()  # Перевод модели в режим оценки (выключает dropout/batchnorm, если они есть)

with torch.no_grad():  # Отключаем расчет градиентов для экономии памяти и времени
    for images, labels in test_loader:
        test_outputs = model(images)
        # Находим индекс максимального значения в строке — это и есть предсказанный класс
        _, predicted_classes = torch.max(test_outputs, dim=1)

        # Считаем точность (Accuracy)
        correct = (predicted_classes == labels).sum().item()
        total = labels.size(0)
        accuracy = (correct / total) * 100
        for i in range(len(labels)):
            print(labels[i], predicted_classes[i])

print(f"\nТочность (Accuracy) на тестовой выборке: {accuracy:.2f}%")

tensor(7) tensor(7)
tensor(2) tensor(2)
tensor(1) tensor(1)
tensor(0) tensor(0)
tensor(4) tensor(4)
tensor(1) tensor(1)
tensor(4) tensor(4)
tensor(9) tensor(9)
tensor(5) tensor(5)
tensor(9) tensor(9)
tensor(0) tensor(0)
tensor(6) tensor(6)
tensor(9) tensor(9)
tensor(0) tensor(0)
tensor(1) tensor(1)
tensor(5) tensor(5)
tensor(9) tensor(9)
tensor(7) tensor(7)
tensor(3) tensor(3)
tensor(4) tensor(4)
tensor(9) tensor(9)
tensor(6) tensor(6)
tensor(6) tensor(6)
tensor(5) tensor(5)
tensor(4) tensor(4)
tensor(0) tensor(0)
tensor(7) tensor(7)
tensor(4) tensor(4)
tensor(0) tensor(0)
tensor(1) tensor(1)
tensor(3) tensor(3)
tensor(1) tensor(1)
tensor(3) tensor(3)
tensor(4) tensor(4)
tensor(7) tensor(7)
tensor(2) tensor(2)
tensor(7) tensor(7)
tensor(1) tensor(1)
tensor(2) tensor(5)
tensor(1) tensor(1)
tensor(1) tensor(1)
tensor(7) tensor(7)
tensor(4) tensor(4)
tensor(2) tensor(2)
tensor(3) tensor(3)
tensor(5) tensor(5)
tensor(1) tensor(1)
tensor(2) tensor(2)
tensor(4) tensor(4)
tensor(4) tensor(4)
